In [3]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.svm import SVR

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

import mlflow

In [4]:
# set the dagshub tracking server

mlflow.set_tracking_uri("https://dagshub.com/Anubhavspeaks01/uber-demand-prediction.mlflow")

In [5]:
import dagshub
dagshub.init(repo_owner='Anubhavspeaks01', repo_name='uber-demand-prediction', mlflow=True)

Accessing as Anubhavspeaks01

Initialized MLflow to track repo "Anubhavspeaks01/uber-demand-prediction"

Repository Anubhavspeaks01/uber-demand-prediction initialized!

In [6]:
# load the training and test data

train_data_path = "../data/processed/train.csv"
test_data_path = "../data/processed/test.csv"

train_df = pd.read_csv(train_data_path, parse_dates=["tpep_pickup_datetime"]).set_index("tpep_pickup_datetime")

test_df = pd.read_csv(test_data_path, parse_dates=["tpep_pickup_datetime"]).set_index("tpep_pickup_datetime")

train_df

,lag_1,lag_2,lag_3,lag_4,region,total_pickups,avg_pickups,day_of_week
tpep_pickup_datetime,,,,,,,,
2016-01-01 01:00:00,160.0,149.0,120.0,58.0,0,187,161.0,4
2016-01-01 01:15:00,187.0,160.0,149.0,120.0,0,194,175.0,4
2016-01-01 01:30:00,194.0,187.0,160.0,149.0,0,180,177.0,4
2016-01-01 01:45:00,180.0,194.0,187.0,160.0,0,197,185.0,4
2016-01-01 02:00:00,197.0,180.0,194.0,187.0,0,185,185.0,4
...,...,...,...,...,...,...,...,...
2016-02-29 22:45:00,15.0,9.0,11.0,11.0,29,12,12.0,0
2016-02-29 23:00:00,12.0,15.0,9.0,11.0,29,17,14.0,0
2016-02-29 23:15:00,17.0,12.0,15.0,9.0,29,15,14.0,0


In [7]:
# missing value in training data

train_df.isna().sum()

lag_1            0
lag_2            0
lag_3            0
lag_4            0
region           0
total_pickups    0
avg_pickups      0
day_of_week      0
dtype: int64

In [8]:
# missing values in the test data

test_df.isna().sum()

lag_1            0
lag_2            0
lag_3            0
lag_4            0
region           0
total_pickups    0
avg_pickups      0
day_of_week      0
dtype: int64

In [9]:
# make X_train and y_train

X_train = train_df.drop(columns=["total_pickups"])

y_train = train_df["total_pickups"]

In [10]:
X_train.head()

,lag_1,lag_2,lag_3,lag_4,region,avg_pickups,day_of_week
tpep_pickup_datetime,,,,,,,
2016-01-01 01:00:00,160.0,149.0,120.0,58.0,0,161.0,4
2016-01-01 01:15:00,187.0,160.0,149.0,120.0,0,175.0,4
2016-01-01 01:30:00,194.0,187.0,160.0,149.0,0,177.0,4
2016-01-01 01:45:00,180.0,194.0,187.0,160.0,0,185.0,4
2016-01-01 02:00:00,197.0,180.0,194.0,187.0,0,185.0,4


In [11]:
# make X_test and y_test

X_test = test_df.drop(columns=["total_pickups"])

y_test = test_df["total_pickups"]

In [12]:
X_test.head()

,lag_1,lag_2,lag_3,lag_4,region,avg_pickups,day_of_week
tpep_pickup_datetime,,,,,,,
2016-03-01 00:00:00,36.0,44.0,31.0,29.0,0,39.0,1
2016-03-01 00:15:00,41.0,36.0,44.0,31.0,0,37.0,1
2016-03-01 00:30:00,35.0,41.0,36.0,44.0,0,41.0,1
2016-03-01 00:45:00,47.0,35.0,41.0,36.0,0,38.0,1
2016-03-01 01:00:00,34.0,47.0,35.0,41.0,0,35.0,1


In [13]:
from sklearn import set_config

set_config(transform_output="pandas")

In [14]:
# encode the data

encoder = ColumnTransformer([
    ("ohe", OneHotEncoder(drop="first",sparse_output=False), ["region","day_of_week"])
], remainder="passthrough", n_jobs=-1,force_int_remainder_cols=False)


In [15]:
encoder

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('ohe', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",-1
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{featu

In [18]:
# encode the train and test data

X_train_encoded = encoder.fit_transform(X_train)
X_test_encoded = encoder.transform(X_test)

/Users/anubhavsingh/Uber_Demand_Prediction/.conda/lib/python3.14/site-packages/sklearn/compose/_column_transformer.py:978: FutureWarning: The parameter `force_int_remainder_cols` is deprecated and will be removed in 1.9. It has no effect. Leave it to its default value to avoid this warning.
  warnings.warn(


In [19]:
import optuna
import tqdm 

/Users/anubhavsingh/Uber_Demand_Prediction/.conda/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [20]:
# set the experiment

mlflow.set_experiment("Model Selection")

2026/03/03 23:06:08 INFO mlflow.tracking.fluent: Experiment with name 'Model Selection' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/55f071ca609e4033bf5000c54815bd97', creation_time=1772559368994, experiment_id='1', last_update_time=1772559368994, lifecycle_stage='active', name='Model Selection', tags={}, workspace='default'>

In [21]:
def objective(trial):
    # start the child run
    with mlflow.start_run(nested=True) as child:
        
        # model name search space
        list_of_models = ["LR", "RF", "GBR", "XGBR"]
        model_name = trial.suggest_categorical("model_name", list_of_models)
    
        if model_name == "LR":
            model = LinearRegression()
    
        elif model_name == "RF":
            n_estimators_rf = trial.suggest_int("n_estimators_rf",10,100,step=10)
            max_depth_rf = trial.suggest_int("max_depth_rf",3,10)
            model = RandomForestRegressor(n_estimators=n_estimators_rf, 
                                          max_depth=max_depth_rf, 
                                          random_state=42, n_jobs=-1)
    
        elif model_name == "GBR":
            n_estimators_gb = trial.suggest_int("n_estimators_gb",10,100,step=10)
            learning_rate_gb = trial.suggest_float("learning_rate_gb",1e-4,1e-1, log=True)
            model = GradientBoostingRegressor(n_estimators=n_estimators_gb, 
                                              learning_rate=learning_rate_gb,
                                             random_state=42)
    
        elif model_name == "XGBR":
            n_estimators_xgb = trial.suggest_int("n_estimators_xgb",10,100,step=10)
            learning_rate_xgb = trial.suggest_float("learning_rate_xgb",1e-4,1e-1, log=True)
            max_depth_xgb = trial.suggest_int("max_depth_xgb",3,10)
            model = XGBRegressor(n_estimators=n_estimators_xgb,
                                learning_rate=learning_rate_xgb,
                                max_depth=max_depth_xgb)
    
        # log the model name
        mlflow.log_param("model_name",model_name)
        
        # log the model parameters
        mlflow.log_params(model.get_params())
        
        # fit on the data
        model.fit(X_train_encoded,y_train)
    
        # get the predictions
        y_pred = model.predict(X_test_encoded)
    
        # calculate the loss
        loss = mean_absolute_percentage_error(y_test, y_pred)
    
        # log the metric
        mlflow.log_metric("MAPE",loss)
        return loss


In [22]:
# optimize the objective function

with mlflow.start_run(run_name="best_model", nested=True) as parent:

    # create a study object
    study = optuna.create_study(study_name="model_selection", direction="minimize")
    # optimize the objective function
    study.optimize(func=objective, n_trials=50, n_jobs=-1)
    
    # log the best parameters
    mlflow.log_params(study.best_params)
    # log the best error value
    mlflow.log_metric("Best_MAPE", study.best_value)

[I 2026-03-03 23:06:34,903] A new study created in memory with name: model_selection


🏃 View run adaptable-shad-254 at: https://dagshub.com/Anubhavspeaks01/uber-demand-prediction.mlflow/#/experiments/1/runs/4dea72a3c8104489b0efc81733b10527
🧪 View experiment at: https://dagshub.com/Anubhavspeaks01/uber-demand-prediction.mlflow/#/experiments/1


[I 2026-03-03 23:07:06,579] Trial 0 finished with value: 0.17680868983106915 and parameters: {'model_name': 'RF', 'n_estimators_rf': 80, 'max_depth_rf': 7}. Best is trial 0 with value: 0.17680868983106915.


🏃 View run painted-auk-191 at: https://dagshub.com/Anubhavspeaks01/uber-demand-prediction.mlflow/#/experiments/1/runs/1fcc41b6ba3d4280be41171f2321740e
🧪 View experiment at: https://dagshub.com/Anubhavspeaks01/uber-demand-prediction.mlflow/#/experiments/1
🏃 View run beautiful-hog-81 at: https://dagshub.com/Anubhavspeaks01/uber-demand-prediction.mlflow/#/experiments/1/runs/fbde3c2bbd9a44958f448b5f6ba1475f
🧪 View experiment at: https://dagshub.com/Anubhavspeaks01/uber-demand-prediction.mlflow/#/experiments/1
🏃 View run industrious-newt-458 at: https://dagshub.com/Anubhavspeaks01/uber-demand-prediction.mlflow/#/experiments/1/runs/b77a8222bc9641ac93c0017d369acdac
🧪 View experiment at: https://dagshub.com/Anubhavspeaks01/uber-demand-prediction.mlflow/#/experiments/1
🏃 View run receptive-zebra-925 at: https://dagshub.com/Anubhavspeaks01/uber-demand-prediction.mlflow/#/experiments/1/runs/87fe58b0f0a9432f91267d7ccb8f36f2
🧪 View experiment at: https://dagshub.com/Anubhavspeaks01/uber-demand-pred

[I 2026-03-03 23:07:17,554] Trial 7 finished with value: 6.074960368426332 and parameters: {'model_name': 'GBR', 'n_estimators_gb': 10, 'learning_rate_gb': 0.008419107163414175}. Best is trial 0 with value: 0.17680868983106915.
[I 2026-03-03 23:07:18,555] Trial 6 finished with value: 0.07934790285463009 and parameters: {'model_name': 'LR'}. Best is trial 6 with value: 0.07934790285463009.
[I 2026-03-03 23:07:19,566] Trial 9 finished with value: 1.1672030300157346 and parameters: {'model_name': 'GBR', 'n_estimators_gb': 30, 'learning_rate_gb': 0.06345398142589752}. Best is trial 6 with value: 0.07934790285463009.
[I 2026-03-03 23:07:20,570] Trial 3 finished with value: 2.086867570877075 and parameters: {'model_name': 'XGBR', 'n_estimators_xgb': 100, 'learning_rate_xgb': 0.011737315506753757, 'max_depth_xgb': 6}. Best is trial 6 with value: 0.07934790285463009.
[I 2026-03-03 23:07:21,594] Trial 4 finished with value: 3.3144915103912354 and parameters: {'model_name': 'XGBR', 'n_estimators

...

🏃 View run suave-hog-334 at: https://dagshub.com/Anubhavspeaks01/uber-demand-prediction.mlflow/#/experiments/1/runs/6f25c02111c1410292e5b64cce954e7a
🧪 View experiment at: https://dagshub.com/Anubhavspeaks01/uber-demand-prediction.mlflow/#/experiments/1
🏃 View run languid-roo-4 at: https://dagshub.com/Anubhavspeaks01/uber-demand-prediction.mlflow/#/experiments/1/runs/5b2801b3d6844c9c826ee920753ba6fd
🧪 View experiment at: https://dagshub.com/Anubhavspeaks01/uber-demand-prediction.mlflow/#/experiments/1
🏃 View run skillful-grub-95 at: https://dagshub.com/Anubhavspeaks01/uber-demand-prediction.mlflow/#/experiments/1/runs/2bbca4e4613540069b0adaf4190b3459
🧪 View experiment at: https://dagshub.com/Anubhavspeaks01/uber-demand-prediction.mlflow/#/experiments/1
🏃 View run persistent-perch-98 at: https://dagshub.com/Anubhavspeaks01/uber-demand-prediction.mlflow/#/experiments/1/runs/71b75de3278f4caaaefff53d54280429
🧪 View experiment at: https://dagshub.com/Anubhavspeaks01/uber-demand-prediction.ml

[I 2026-03-03 23:11:16,561] Trial 43 finished with value: 6.571044921875 and parameters: {'model_name': 'XGBR', 'n_estimators_xgb': 10, 'learning_rate_xgb': 0.00010402349747662487, 'max_depth_xgb': 3}. Best is trial 6 with value: 0.07934790285463009.


🏃 View run funny-gull-190 at: https://dagshub.com/Anubhavspeaks01/uber-demand-prediction.mlflow/#/experiments/1/runs/aaf3309bb89e4968848819ccedc29131
🧪 View experiment at: https://dagshub.com/Anubhavspeaks01/uber-demand-prediction.mlflow/#/experiments/1
🏃 View run invincible-grouse-1 at: https://dagshub.com/Anubhavspeaks01/uber-demand-prediction.mlflow/#/experiments/1/runs/84d42b3c231e4f9aab50a33605c28034
🧪 View experiment at: https://dagshub.com/Anubhavspeaks01/uber-demand-prediction.mlflow/#/experiments/1


[I 2026-03-03 23:11:19,558] Trial 44 finished with value: 6.569861888885498 and parameters: {'model_name': 'XGBR', 'n_estimators_xgb': 10, 'learning_rate_xgb': 0.00012312566550149118, 'max_depth_xgb': 3}. Best is trial 6 with value: 0.07934790285463009.
[I 2026-03-03 23:11:20,570] Trial 45 finished with value: 6.562217712402344 and parameters: {'model_name': 'XGBR', 'n_estimators_xgb': 10, 'learning_rate_xgb': 0.0002466465114207907, 'max_depth_xgb': 3}. Best is trial 6 with value: 0.07934790285463009.
[I 2026-03-03 23:11:21,563] Trial 46 finished with value: 6.571047782897949 and parameters: {'model_name': 'XGBR', 'n_estimators_xgb': 10, 'learning_rate_xgb': 0.00010397735071606895, 'max_depth_xgb': 3}. Best is trial 6 with value: 0.07934790285463009.
[I 2026-03-03 23:11:22,563] Trial 47 finished with value: 6.569599628448486 and parameters: {'model_name': 'XGBR', 'n_estimators_xgb': 10, 'learning_rate_xgb': 0.000127358934952697, 'max_depth_xgb': 3}. Best is trial 6 with value: 0.079347

🏃 View run best_model at: https://dagshub.com/Anubhavspeaks01/uber-demand-prediction.mlflow/#/experiments/1/runs/1d55608b95d04696828cd35f71932067
🧪 View experiment at: https://dagshub.com/Anubhavspeaks01/uber-demand-prediction.mlflow/#/experiments/1


In [23]:
# best value

study.best_value

0.07934790285463009

In [24]:
# best parameters

study.best_params

{'model_name': 'LR'}

In [25]:
# model value counts

study.trials_dataframe()['params_model_name'].value_counts()

params_model_name
LR      18
XGBR    11
GBR     11
RF      10
Name: count, dtype: int64

In [28]:
from optuna.visualization import (
    plot_optimization_history, 
    plot_parallel_coordinate, 
    plot_param_importances
)

In [29]:
!pip install plotly

In [32]:
# train the linear regression model

lr = LinearRegression()

lr.fit(X_train_encoded, y_train)

# get predictions
y_pred_train = lr.predict(X_train_encoded) 
y_pred_test = lr.predict(X_test_encoded)

# loss

mape_train = mean_absolute_percentage_error(y_train, y_pred_train)
mape_test = mean_absolute_percentage_error(y_test, y_pred_test)

print("The training error is ", mape_train)
print("The test error is ", mape_test)

The training error is  0.08778013304566377
The test error is  0.07934790285463009


In [33]:
lr.coef_

array([-2.33737604,  0.71512405, -0.55601505, -1.25311068, -3.20463231,
       -0.86685973, -2.79925402, -3.62516859,  0.41386463, -2.9376376 ,
       -1.97624678, -3.75050442,  0.51806283, -2.54033388, -2.43297463,
        0.47632075,  0.61254786, -4.7417372 , -2.03077217, -1.26960984,
       -4.03690273, -2.08863167, -1.0414428 ,  0.73561736, -0.99999442,
       -0.85944985, -2.43098478,  0.67112238,  0.57385071, -0.11719951,
       -0.28045898, -0.37180749, -0.5238324 , -0.4233113 , -0.34045774,
       -0.54170892, -0.36264553, -0.2493965 , -0.31905518,  2.4912456 ])

In [34]:
def tune_ridge(trial):
    # hyperparameter space
    alpha = trial.suggest_float("alpha",30,100)
    
    # make the model object
    ridge = Ridge(alpha=alpha, random_state=42)
    
    # train the model
    ridge.fit(X_train_encoded, y_train)
    
    # get predictions
    y_pred = ridge.predict(X_test_encoded)
    
    # calculate loss
    loss = mean_absolute_percentage_error(y_test, y_pred)

    return loss

In [35]:
# create study

study = optuna.create_study(study_name="tune_model", direction="minimize")

[I 2026-03-03 23:15:41,116] A new study created in memory with name: tune_model


In [36]:
# optimize

study.optimize(func=tune_ridge, n_trials=100, n_jobs=-1, show_progress_bar=True)

Best trial: 14. Best value: 0.0791642:  18%|███████▋                                   | 18/100 [00:00<00:01, 41.87it/s]

[I 2026-03-03 23:15:55,905] Trial 0 finished with value: 0.07921058672920982 and parameters: {'alpha': 66.50078155882879}. Best is trial 0 with value: 0.07921058672920982.
[I 2026-03-03 23:15:55,926] Trial 1 finished with value: 0.07917216689141483 and parameters: {'alpha': 93.56000769552267}. Best is trial 1 with value: 0.07917216689141483.
[I 2026-03-03 23:15:55,932] Trial 7 finished with value: 0.07921194072062233 and parameters: {'alpha': 65.62034294367048}. Best is trial 1 with value: 0.07917216689141483.
[I 2026-03-03 23:15:55,934] Trial 5 finished with value: 0.07916665471768947 and parameters: {'alpha': 97.90282135743836}. Best is trial 5 with value: 0.07916665471768947.
[I 2026-03-03 23:15:55,935] Trial 3 finished with value: 0.0792610042537362 and parameters: {'alpha': 37.3427493794714}. Best is trial 5 with value: 0.07916665471768947.
[I 2026-03-03 23:15:55,935] Trial 6 finished with value: 0.07918211308441897 and parameters: {'alpha': 86.10911170147904}. Best is trial 5 wit

[I 2026-03-03 23:15:56,098] Trial 19 finished with value: 0.0791827815198209 and parameters: {'alpha': 85.61939200851275}. Best is trial 14 with value: 0.0791641655894888.
[I 2026-03-03 23:15:56,111] Trial 21 finished with value: 0.07918693662601725 and parameters: {'alpha': 82.62079237741276}. Best is trial 14 with value: 0.0791641655894888.
[I 2026-03-03 23:15:56,128] Trial 20 finished with value: 0.07918561771932539 and parameters: {'alpha': 83.56511406834807}. Best is trial 14 with value: 0.0791641655894888.
[I 2026-03-03 23:15:56,139] Trial 22 finished with value: 0.07918669276519878 and parameters: {'alpha': 82.79473121612709}. Best is trial 14 with value: 0.0791641655894888.
[I 2026-03-03 23:15:56,142] Trial 23 finished with value: 0.0791875221387122 and parameters: {'alpha': 82.20496825076945}. Best is trial 14 with value: 0.0791641655894888.
[I 2026-03-03 23:15:56,151] Trial 24 finished with value: 0.07918358140800967 and parameters: {'alpha': 85.03594011849569}. Best is trial

[I 2026-03-03 23:15:56,321] Trial 37 finished with value: 0.07920323547976689 and parameters: {'alpha': 71.37788487952209}. Best is trial 14 with value: 0.0791641655894888.
[I 2026-03-03 23:15:56,325] Trial 38 finished with value: 0.07920348636057263 and parameters: {'alpha': 71.20928740366095}. Best is trial 14 with value: 0.0791641655894888.
[I 2026-03-03 23:15:56,341] Trial 39 finished with value: 0.07922220093357538 and parameters: {'alpha': 59.13063496091481}. Best is trial 14 with value: 0.0791641655894888.
[I 2026-03-03 23:15:56,353] Trial 40 finished with value: 0.07920723955234298 and parameters: {'alpha': 68.70366879656828}. Best is trial 14 with value: 0.0791641655894888.
[I 2026-03-03 23:15:56,355] Trial 41 finished with value: 0.07922003912026787 and parameters: {'alpha': 60.47624505152477}. Best is trial 14 with value: 0.0791641655894888.
[I 2026-03-03 23:15:56,363] Trial 42 finished with value: 0.07920524552783303 and parameters: {'alpha': 70.03009133561477}. Best is tri

Best trial: 56. Best value: 0.0791641:  69%|█████████████████████████████▋             | 69/100 [00:01<00:00, 77.28it/s]

[I 2026-03-03 23:15:56,533] Trial 56 finished with value: 0.07916409456914995 and parameters: {'alpha': 99.97636931097053}. Best is trial 56 with value: 0.07916409456914995.
[I 2026-03-03 23:15:56,539] Trial 55 finished with value: 0.07916429651112501 and parameters: {'alpha': 99.81187100635644}. Best is trial 56 with value: 0.07916409456914995.
[I 2026-03-03 23:15:56,559] Trial 57 finished with value: 0.07916453704267415 and parameters: {'alpha': 99.61596670367804}. Best is trial 56 with value: 0.07916409456914995.
[I 2026-03-03 23:15:56,570] Trial 59 finished with value: 0.07916452484650159 and parameters: {'alpha': 99.62589930784239}. Best is trial 56 with value: 0.07916409456914995.
[I 2026-03-03 23:15:56,576] Trial 58 finished with value: 0.07916432759467652 and parameters: {'alpha': 99.78655283418702}. Best is trial 56 with value: 0.07916409456914995.
[I 2026-03-03 23:15:56,598] Trial 60 finished with value: 0.07916451045941524 and parameters: {'alpha': 99.6376162972613}. Best is

Best trial: 80. Best value: 0.0791641:  86%|████████████████████████████████████▉      | 86/100 [00:01<00:00, 76.15it/s]

[I 2026-03-03 23:15:56,718] Trial 69 finished with value: 0.0791670863406222 and parameters: {'alpha': 97.55530581917145}. Best is trial 56 with value: 0.07916409456914995.
[I 2026-03-03 23:15:56,734] Trial 71 finished with value: 0.07917216589157156 and parameters: {'alpha': 93.56077347566725}. Best is trial 56 with value: 0.07916409456914995.
[I 2026-03-03 23:15:56,757] Trial 72 finished with value: 0.07916706956092376 and parameters: {'alpha': 97.56880975762081}. Best is trial 56 with value: 0.07916409456914995.
[I 2026-03-03 23:15:56,770] Trial 74 finished with value: 0.0791719741477185 and parameters: {'alpha': 93.70786969813119}. Best is trial 56 with value: 0.07916409456914995.
[I 2026-03-03 23:15:56,781] Trial 73 finished with value: 0.07917281076082183 and parameters: {'alpha': 93.06699964814244}. Best is trial 56 with value: 0.07916409456914995.
[I 2026-03-03 23:15:56,792] Trial 76 finished with value: 0.07916738115730143 and parameters: {'alpha': 97.31938189565568}. Best is 

Best trial: 80. Best value: 0.0791641: 100%|██████████████████████████████████████████| 100/100 [00:01<00:00, 73.41it/s]

[I 2026-03-03 23:15:56,950] Trial 88 finished with value: 0.07916410557187112 and parameters: {'alpha': 99.96740613505735}. Best is trial 80 with value: 0.07916406685151742.
[I 2026-03-03 23:15:56,951] Trial 87 finished with value: 0.0791648789653811 and parameters: {'alpha': 99.33759625462254}. Best is trial 80 with value: 0.07916406685151742.
[I 2026-03-03 23:15:56,988] Trial 91 finished with value: 0.07917136310396239 and parameters: {'alpha': 94.17888794710079}. Best is trial 80 with value: 0.07916406685151742.
[I 2026-03-03 23:15:56,991] Trial 89 finished with value: 0.07916437407135372 and parameters: {'alpha': 99.74869761633673}. Best is trial 80 with value: 0.07916406685151742.
[I 2026-03-03 23:15:56,991] Trial 90 finished with value: 0.07916527457707347 and parameters: {'alpha': 99.01618939794163}. Best is trial 80 with value: 0.07916406685151742.
[I 2026-03-03 23:15:57,018] Trial 92 finished with value: 0.07917119216648517 and parameters: {'alpha': 94.31145628826495}. Best is

In [37]:
# best parameters

study.best_params

{'alpha': 99.99894929734239}

In [38]:
# best value

study.best_value

0.07916406685151742